# RandomForest HIV Classification with 5-Fold CV

This notebook evaluates the RandomForest model on the HIV dataset using Morgan fingerprints, stratified 5-fold cross-validation, fold-level validation/test splits, and summary statistics (mean, std, variance).

In [ ]:

# If needed in Colab:
!pip -q install rdkit

import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem
RDLogger.DisableLog('rdApp.*')

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve
)
from sklearn.preprocessing import StandardScaler


In [2]:

SEED = 45
N_SPLITS = 5
VAL_SIZE_WITHIN_TEMP = 0.5  # temp set is split equally into val and test
RADIUS = 2
N_BITS = 2048

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

set_seed(SEED)

def smiles_to_morgan_fp(smiles_list, radius=2, n_bits=2048):
    fps = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
            fps.append(np.array(fp, dtype=np.float32))
        else:
            fps.append(np.zeros(n_bits, dtype=np.float32))
    return np.array(fps, dtype=np.float32)

def build_model(random_state):
    from sklearn.ensemble import RandomForestClassifier
    return RandomForestClassifier(
        n_estimators=100,
        random_state=random_state,
        n_jobs=-1
    )

def maybe_scale(X_train, X_val, X_test):
    return X_train, X_val, X_test


def evaluate_fold(model, X_test, y_test):
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    else:
        y_score = model.decision_function(X_test)

    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_score)
    }, y_score


In [ ]:

df = pd.read_csv("https://raw.githubusercontent.com/McahitKutsal/hivcsv/main/HIV7.csv")
df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

X = smiles_to_morgan_fp(df["smiles"], radius=RADIUS, n_bits=N_BITS)
y = df["HIV_active"].astype(int).values

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

fold_results = []
roc_curves = []

for fold, (train_idx, temp_idx) in enumerate(skf.split(X, y), start=1):
    print(f"\n===== FOLD {fold} / {N_SPLITS} =====")
    set_seed(SEED + fold)

    X_train_full, y_train_full = X[train_idx], y[train_idx]
    X_temp, y_temp = X[temp_idx], y[temp_idx]

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp,
        test_size=VAL_SIZE_WITHIN_TEMP,
        stratify=y_temp,
        random_state=SEED + fold
    )

    X_train, X_val, X_test = maybe_scale(X_train_full, X_val, X_test)

    model = build_model(random_state=SEED + fold)
    model.fit(X_train, y_train_full)

    metrics, y_score = evaluate_fold(model, X_test, y_test)
    fold_results.append({
        "fold": fold,
        "test_accuracy": metrics["accuracy"],
        "test_precision": metrics["precision"],
        "test_recall": metrics["recall"],
        "test_f1": metrics["f1"],
        "test_roc_auc": metrics["roc_auc"]
    })

    fpr, tpr, _ = roc_curve(y_test, y_score)
    roc_curves.append((fold, fpr, tpr, metrics["roc_auc"]))

    print({k: round(v, 4) if isinstance(v, float) else v for k, v in fold_results[-1].items()})

results_df = pd.DataFrame(fold_results)
results_df


In [ ]:

summary_rows = []
for metric in ["test_accuracy", "test_precision", "test_recall", "test_f1", "test_roc_auc"]:
    mean_val = results_df[metric].mean()
    std_val = results_df[metric].std(ddof=1)
    var_val = results_df[metric].var(ddof=1)
    summary_rows.append({
        "Metric": metric.replace("test_", "").upper(),
        "Mean": mean_val,
        "Std": std_val,
        "Variance": var_val,
        "Formatted": f"{mean_val:.4f} ± {std_val:.4f}"
    })

summary_df = pd.DataFrame(summary_rows)
summary_df


In [ ]:

plt.figure(figsize=(8, 6))
for fold, fpr, tpr, auc_val in roc_curves:
    plt.plot(fpr, tpr, label=f"Fold {fold} (AUC={auc_val:.3f})")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("RandomForest ROC Curves Across 5 Folds")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:

results_df.to_csv("RandomForest_fold_results.csv", index=False)
summary_df.to_csv("RandomForest_summary_results.csv", index=False)

print("Saved:")
print("- RandomForest_fold_results.csv")
print("- RandomForest_summary_results.csv")


In [ ]:

# =========================================================
# RF DOCKING PREPARATION: OOF PREDICTIONS + TOP-2 CANDIDATES
# =========================================================

from rdkit.Chem import Descriptors, Lipinski, Crippen, QED, Draw

# 1) Build out-of-fold (OOF) prediction table using the same 5-fold protocol
oof_rows = []

for fold, (train_idx, temp_idx) in enumerate(skf.split(X, y), start=1):
    set_seed(SEED + fold)

    X_train_full, y_train_full = X[train_idx], y[train_idx]
    X_temp, y_temp = X[temp_idx], y[temp_idx]
    smiles_temp = df.iloc[temp_idx]["smiles"].reset_index(drop=True)

    _, X_test, _, y_test = train_test_split(
        X_temp, y_temp,
        test_size=VAL_SIZE_WITHIN_TEMP,
        stratify=y_temp,
        random_state=SEED + fold
    )

    # Recover smiles for the selected test split by repeating the same split on smiles
    _, smiles_test, _, _ = train_test_split(
        smiles_temp,
        y_temp,
        test_size=VAL_SIZE_WITHIN_TEMP,
        stratify=y_temp,
        random_state=SEED + fold
    )

    X_train, _, X_test = maybe_scale(X_train_full, X_test, X_test)

    rf_model = build_model(random_state=SEED + fold)
    rf_model.fit(X_train, y_train_full)

    y_prob = rf_model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    fold_df = pd.DataFrame({
        "fold": fold,
        "smiles": list(smiles_test),
        "y_true": y_test,
        "y_pred": y_pred,
        "y_prob": y_prob
    })
    oof_rows.append(fold_df)

oof_pred_df = pd.concat(oof_rows, ignore_index=True)
oof_pred_df = oof_pred_df.drop_duplicates(subset=["smiles"]).reset_index(drop=True)
oof_pred_df.to_csv("RF_oof_predictions.csv", index=False)

print("Saved: RF_oof_predictions.csv")
display(oof_pred_df.head())


# 2) Rank molecules by RF confidence
top_df = (
    oof_pred_df[oof_pred_df["y_pred"] == 1]
    .sort_values(by="y_prob", ascending=False)
    .head(10)
    .reset_index(drop=True)
)
top_df.to_csv("RF_top_10_candidates.csv", index=False)
print("Saved: RF_top_10_candidates.csv  |  Top rows:", len(top_df))
display(top_df)


# 3) Compute descriptors
def compute_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return {
        "MW": Descriptors.MolWt(mol),
        "LogP": Crippen.MolLogP(mol),
        "HBD": Lipinski.NumHDonors(mol),
        "HBA": Lipinski.NumHAcceptors(mol),
        "TPSA": Descriptors.TPSA(mol),
        "RotatableBonds": Lipinski.NumRotatableBonds(mol),
        "QED": QED.qed(mol)
    }

desc_rows = []
for _, row in top_df.iterrows():
    desc = compute_descriptors(row["smiles"])
    if desc is not None:
        desc.update(row.to_dict())
        desc_rows.append(desc)

desc_df = pd.DataFrame(desc_rows)
desc_df.to_csv("RF_top_10_descriptors.csv", index=False)
print("Saved: RF_top_10_descriptors.csv")
display(desc_df.head())


# 4) Drug-likeness filter
filtered_df = desc_df[
    (desc_df["MW"] <= 500) &
    (desc_df["LogP"] <= 5) &
    (desc_df["HBD"] <= 5) &
    (desc_df["HBA"] <= 10)
].copy()

filtered_df["Lipinski_pass"] = True
filtered_df.to_csv("RF_docking_candidates_filtered.csv", index=False)
print("Saved: RF_docking_candidates_filtered.csv")
display(filtered_df)


# 5) Select the final TOP-2 candidates
# Priority: QED first, then RF probability
final_df = filtered_df.sort_values(by=["QED", "y_prob"], ascending=False).head(2).copy()

if len(final_df) < 2:
    print("Warning: fewer than 2 Lipinski-passing molecules were found. Falling back to top-ranked filtered rows.")
    final_df = filtered_df.sort_values(by=["y_prob"], ascending=False).head(2).copy()

final_df.to_csv("RF_final_2_candidates.csv", index=False)
final_df["smiles"].to_csv("RF_docking_input.smi", index=False, header=False)

print("Saved: RF_final_2_candidates.csv")
print("Saved: RF_docking_input.smi")
display(final_df[["smiles", "y_prob", "QED", "MW", "LogP"]])


# 6) Draw only the final 2 molecules
best_smiles = final_df["smiles"].tolist()
best_mols = [Chem.MolFromSmiles(sm) for sm in best_smiles]

legends = []
for i, (_, row) in enumerate(final_df.iterrows(), start=1):
    legends.append(f"RF Molecule {i}\nProb={row['y_prob']:.3f} | QED={row['QED']:.3f}")

img = Draw.MolsToGridImage(
    best_mols,
    molsPerRow=2,
    subImgSize=(320, 320),
    legends=legends
)
display(img)

print("\nSummary")
print("-------")
print(f"Total OOF predictions          : {len(oof_pred_df)}")
print(f"Unique ranked molecules        : {oof_pred_df['smiles'].nunique()}")
print(f"Top active candidates exported : {len(top_df)}")
print(f"Lipinski-passing candidates    : {len(filtered_df)}")
print(f"Final docking candidates       : {len(final_df)}")


In [ ]:
# =========================================================
# RF DOCKING PREPARATION: OOF PREDICTIONS + TOP-2 CANDIDATES
# (FULL SMILES DISPLAY FIXED VERSION)
# =========================================================

import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, Crippen, QED, Draw

# 🔴 SMILES kesilmesini engelle
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# 1) Build out-of-fold (OOF) prediction table
oof_rows = []

for fold, (train_idx, temp_idx) in enumerate(skf.split(X, y), start=1):
    set_seed(SEED + fold)

    X_train_full, y_train_full = X[train_idx], y[train_idx]
    X_temp, y_temp = X[temp_idx], y[temp_idx]
    smiles_temp = df.iloc[temp_idx]["smiles"].reset_index(drop=True)

    _, X_test, _, y_test = train_test_split(
        X_temp, y_temp,
        test_size=VAL_SIZE_WITHIN_TEMP,
        stratify=y_temp,
        random_state=SEED + fold
    )

    _, smiles_test, _, _ = train_test_split(
        smiles_temp,
        y_temp,
        test_size=VAL_SIZE_WITHIN_TEMP,
        stratify=y_temp,
        random_state=SEED + fold
    )

    X_train, _, X_test = maybe_scale(X_train_full, X_test, X_test)

    rf_model = build_model(random_state=SEED + fold)
    rf_model.fit(X_train, y_train_full)

    y_prob = rf_model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    fold_df = pd.DataFrame({
        "fold": fold,
        "smiles": list(smiles_test),
        "y_true": y_test,
        "y_pred": y_pred,
        "y_prob": y_prob
    })
    oof_rows.append(fold_df)

oof_pred_df = pd.concat(oof_rows, ignore_index=True)
oof_pred_df = oof_pred_df.drop_duplicates(subset=["smiles"]).reset_index(drop=True)
oof_pred_df.to_csv("RF_oof_predictions.csv", index=False)

print("Saved: RF_oof_predictions.csv")
display(oof_pred_df.head())


# 2) Rank molecules
top_df = (
    oof_pred_df[oof_pred_df["y_pred"] == 1]
    .sort_values(by="y_prob", ascending=False)
    .head(10)
    .reset_index(drop=True)
)
top_df.to_csv("RF_top_10_candidates.csv", index=False)
print("Saved: RF_top_10_candidates.csv  |  Top rows:", len(top_df))
display(top_df)


# 3) Compute descriptors
def compute_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return {
        "MW": Descriptors.MolWt(mol),
        "LogP": Crippen.MolLogP(mol),
        "HBD": Lipinski.NumHDonors(mol),
        "HBA": Lipinski.NumHAcceptors(mol),
        "TPSA": Descriptors.TPSA(mol),
        "RotatableBonds": Lipinski.NumRotatableBonds(mol),
        "QED": QED.qed(mol)
    }

desc_rows = []
for _, row in top_df.iterrows():
    desc = compute_descriptors(row["smiles"])
    if desc is not None:
        desc.update(row.to_dict())
        desc_rows.append(desc)

desc_df = pd.DataFrame(desc_rows)
desc_df.to_csv("RF_top_10_descriptors.csv", index=False)
print("Saved: RF_top_10_descriptors.csv")
display(desc_df.head())


# 4) Drug-likeness filter
filtered_df = desc_df[
    (desc_df["MW"] <= 500) &
    (desc_df["LogP"] <= 5) &
    (desc_df["HBD"] <= 5) &
    (desc_df["HBA"] <= 10)
].copy()

filtered_df["Lipinski_pass"] = True
filtered_df.to_csv("RF_docking_candidates_filtered.csv", index=False)
print("Saved: RF_docking_candidates_filtered.csv")
display(filtered_df)


# 5) Final TOP-2
final_df = filtered_df.sort_values(by=["QED", "y_prob"], ascending=False).head(2).copy()

if len(final_df) < 2:
    print("Warning: fallback used.")
    final_df = filtered_df.sort_values(by=["y_prob"], ascending=False).head(2).copy()

final_df.to_csv("RF_final_2_candidates.csv", index=False)
final_df["smiles"].to_csv("RF_docking_input.smi", index=False, header=False)

print("Saved: RF_final_2_candidates.csv")
print("Saved: RF_docking_input.smi")

display(final_df[["smiles", "y_prob", "QED", "MW", "LogP"]])


# 🔥 FULL SMILES PRINT (NO TRUNCATION)
print("\nFULL SMILES (FINAL 2):")
for i, smi in enumerate(final_df["smiles"], start=1):
    print(f"{i}. {smi}")


# 6) Draw molecules
best_smiles = final_df["smiles"].tolist()
best_mols = [Chem.MolFromSmiles(sm) for sm in best_smiles]

legends = []
for i, (_, row) in enumerate(final_df.iterrows(), start=1):
    legends.append(f"RF Molecule {i}\nProb={row['y_prob']:.3f} | QED={row['QED']:.3f}")

img = Draw.MolsToGridImage(
    best_mols,
    molsPerRow=2,
    subImgSize=(320, 320),
    legends=legends
)
display(img)


print("\nSummary")
print("-------")
print(f"Total OOF predictions          : {len(oof_pred_df)}")
print(f"Unique ranked molecules        : {oof_pred_df['smiles'].nunique()}")
print(f"Top active candidates exported : {len(top_df)}")
print(f"Lipinski-passing candidates    : {len(filtered_df)}")
print(f"Final docking candidates       : {len(final_df)}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')